<a href="https://colab.research.google.com/github/Rishii077/AI-Lab-Assignments/blob/main/Experiment_1_Text_to_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q google-genai

In [4]:
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")

print("API key loaded successfully!")

API key loaded successfully!


In [5]:
from google import genai

client = genai.Client(api_key=API_KEY)

print("Gemini connected successfully!")

Gemini connected successfully!


In [6]:
import sqlite3

# Create a database
conn = sqlite3.connect("students.db")

# Create a cursor
cursor = conn.cursor()

# Create the students table
cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    age INTEGER,
    marks INTEGER
)
""")

# Remove old data if the cell is run again
cursor.execute("DELETE FROM students")

# Student data
students = [
    (1, "Rahul", "CSE", 21, 85),
    (2, "Priya", "ECE", 22, 91),
    (3, "Arjun", "CSE", 20, 78),
    (4, "Sneha", "IT", 21, 88),
    (5, "Kiran", "ECE", 22, 95)
]

# Insert the data
cursor.executemany(
    "INSERT INTO students VALUES (?, ?, ?, ?, ?)",
    students
)

# Save the database
conn.commit()

print("Database created successfully!")

Database created successfully!


In [7]:

cursor.execute("SELECT * FROM students")

results = cursor.fetchall()

for student in results:
    print(student)

(1, 'Rahul', 'CSE', 21, 85)
(2, 'Priya', 'ECE', 22, 91)
(3, 'Arjun', 'CSE', 20, 78)
(4, 'Sneha', 'IT', 21, 88)
(5, 'Kiran', 'ECE', 22, 95)


In [8]:
cursor.execute("""
SELECT name, marks
FROM students
WHERE marks > 90
""")

results = cursor.fetchall()

for student in results:
    print(student)

('Priya', 91)
('Kiran', 95)


In [10]:
def generate_sql(question):

    prompt = f"""
You are a SQL expert.

We have a SQLite database with a table called students.

The table has these columns:
- id
- name
- department
- age
- marks

Convert the user's question into a SQL query.

Return ONLY the SQL query.
Do not add explanations.
Do not use markdown code blocks.

User question:
{question}
"""

    interaction = client.interactions.create(
        model="gemini-3.6-flash",
        input=prompt
    )

    sql_query = interaction.output_text.strip()

    return sql_query


question = "Who scored more than 90 marks?"

sql = generate_sql(question)

print("Generated SQL:")
print(sql)

Generated SQL:
SELECT name FROM students WHERE marks > 90;


In [11]:
def run_sql(sql_query):

    try:
        cursor.execute(sql_query)

        results = cursor.fetchall()

        return results

    except Exception as e:
        return f"Error: {e}"


results = run_sql(sql)

print("Database Result:")
print(results)

Database Result:
[('Priya',), ('Kiran',)]


In [12]:
question = "What is the average marks of all students?"

sql = generate_sql(question)

print("Generated SQL:")
print(sql)

results = run_sql(sql)

print("\nDatabase Result:")
print(results)

Generated SQL:
SELECT AVG(marks) FROM students;

Database Result:
[(87.4,)]


In [13]:
question = "Which students are from the CSE department?"

sql = generate_sql(question)

print("Generated SQL:")
print(sql)

results = run_sql(sql)

print("\nDatabase Result:")
print(results)

Generated SQL:
SELECT * FROM students WHERE department = 'CSE';

Database Result:
[(1, 'Rahul', 'CSE', 21, 85), (3, 'Arjun', 'CSE', 20, 78)]


In [14]:
def display_results(results):

    print("Answer:")

    if not results:
        print("No results found.")
        return

    for row in results:
        print(" - " + " | ".join(str(value) for value in row))

In [15]:
question = "Who scored more than 90 marks?"

sql = generate_sql(question)

print("Question:")
print(question)

print("\nGenerated SQL:")
print(sql)

results = run_sql(sql)

print()

display_results(results)

Question:
Who scored more than 90 marks?

Generated SQL:
SELECT name FROM students WHERE marks > 90;

Answer:
 - Priya
 - Kiran


In [16]:
def ask_database(question):

    print("Question:")
    print(question)

    sql = generate_sql(question)

    print("\nGenerated SQL:")
    print(sql)

    results = run_sql(sql)

    print("\nAnswer:")

    if isinstance(results, str):
        print(results)
        return

    if not results:
        print("No results found.")
        return

    for row in results:
        print(" - " + " | ".join(str(value) for value in row))

In [17]:
ask_database("Who scored more than 90 marks?")

Question:
Who scored more than 90 marks?

Generated SQL:
SELECT name FROM students WHERE marks > 90;

Answer:
 - Priya
 - Kiran


In [18]:
ask_database("Which students are from the CSE department?")

Question:
Which students are from the CSE department?

Generated SQL:
SELECT * FROM students WHERE department = 'CSE';

Answer:
 - 1 | Rahul | CSE | 21 | 85
 - 3 | Arjun | CSE | 20 | 78


In [19]:
def get_database_schema():

    schema = """
    Database: students

    Table: students

    Columns:
    - id: student ID
    - name: student name
    - department: student's department
    - age: student's age
    - marks: student's marks
    """

    return schema


schema = get_database_schema()

print("Retrieved Database Schema:")
print(schema)

Retrieved Database Schema:

    Database: students

    Table: students

    Columns:
    - id: student ID
    - name: student name
    - department: student's department
    - age: student's age
    - marks: student's marks
    


In [20]:
def text_to_sql_system(question):

    # Step 1: Retrieve database schema
    schema = get_database_schema()

    # Step 2: Ask Gemini to generate SQL
    prompt = f"""
You are a SQL expert.

Use the following database schema:

{schema}

Convert the user's question into a SQLite SQL query.

Return ONLY the SQL query.
Do not provide explanations.
Do not use markdown code blocks.

User question:
{question}
"""

    interaction = client.interactions.create(
        model="gemini-3.6-flash",
        input=prompt
    )

    sql_query = interaction.output_text.strip()

    # Step 3: Execute SQL
    results = run_sql(sql_query)

    # Step 4: Display everything
    print("User Question:")
    print(question)

    print("\nGenerated SQL:")
    print(sql_query)

    print("\nAnswer:")

    if isinstance(results, str):
        print(results)
        return

    if not results:
        print("No results found.")
        return

    for row in results:
        print(" - " + " | ".join(str(value) for value in row))

In [21]:
text_to_sql_system("Who scored more than 90 marks?")

User Question:
Who scored more than 90 marks?

Generated SQL:
SELECT name FROM students WHERE marks > 90;

Answer:
 - Priya
 - Kiran


In [22]:
text_to_sql_system("What is the average marks of all students?")

User Question:
What is the average marks of all students?

Generated SQL:
SELECT AVG(marks) FROM students;

Answer:
 - 87.4
